[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_seminar_tests/MFM_ch1_seminar_tests.ipynb)

# MFM_ch1_seminar_tests

Seminar 1 inference and model checks: classical vs heteroskedasticity-robust Ljung-Box, expanding-window kurtosis vs simulated Student-t bands, Hill plots on both tails (BET-TR, Bitcoin), leverage correlations and the leverage regression with Newey-West errors, the BET-TR vs S&P 500 Sharpe difference test with non-synchronous trading, a GARCH(1,1)-t simulation check of stylised facts, and the BET-TR autocorrelation before and after the 2020 FTSE Russell reclassification.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')
import sys
if 'google.colab' in sys.modules:
    !pip -q install arch
import statsmodels.api as sm

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # de la introducerea leului nou (RON), 1 iulie 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# cotatii OTC (aur): sesiunile partiale de sambata/duminica rup randamentul vineri -> luni,
# deci pastram doar zilele lucratoare (aceeasi conventie ca in Capitolul 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """OHLCV zilnic cu coloane Open/High/Low/Close/Volume (Close = pretul folosit)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Pretul de inchidere, folosit in toate graficele."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
SEED = 42

HILL_FRACS_TAILS = np.round(np.arange(0.01, 0.1001, 0.0025), 4)

BLUE_CHIPS = ['TLV.RO', 'SNP.RO', 'BRD.RO', 'TGN.RO', 'FP.RO', 'SNG.RO', 'H2O.RO', 'SNN.RO', 'EL.RO', 'DIGI.RO', 'TEL.RO']

def hill_estimator(x, k):
    """alpha_hat(k) = [ (1/k) sum_{i=1..k} ln(X_(i) / X_(k+1)) ]^{-1}, X_(1) >= X_(2) >= ..."""
    x = np.sort(np.asarray(x)[np.asarray(x) > 0])[::-1]
    return 1.0 / np.mean(np.log(x[:k] / x[k]))


def leverage_corr(r, K=20):
    ks = np.arange(-K, K + 1)
    a = r.abs()
    return ks, np.array([r.corr(a.shift(-k)) for k in ks])     # corr(r_t, |r_{t+k}|)


def rho_robust(r):
    e = (r - r.mean()).values
    s2 = np.sum(e ** 2)
    rho = np.sum(e[1:] * e[:-1]) / s2
    se = np.sqrt(np.sum(e[1:] ** 2 * e[:-1] ** 2)) / s2
    return rho, se


def _kurt_path(x, ends):
    """Excesul de kurtosis pe ferestre extinse [0, e) calculat din sume cumulate (vectorizat pe coloane)."""
    c1, c2, c3, c4 = (np.cumsum(x ** p, axis=0) for p in (1, 2, 3, 4))
    n = ends[:, None].astype(float)
    m1, m2, m3, m4 = (c[ends - 1] / n for c in (c1, c2, c3, c4))
    var = m2 - m1 ** 2
    mu4 = m4 - 4 * m1 * m3 + 6 * m1 ** 2 * m2 - 3 * m1 ** 4
    return mu4 / var ** 2 - 3

## Analysis

In [ ]:
def robust_lb_table(assets=('sp500', 'bettr', 'eurron'), m_q=10, m_band=20):
    rows = []
    for a in assets:
        x = (rets[a] - rets[a].mean()).values
        T, s2 = len(x), np.sum(x ** 2)
        ks = np.arange(1, m_band + 1)
        rho = np.array([np.sum(x[k:] * x[:-k]) / s2 for k in ks])
        tau = np.array([np.sum(x[k:] ** 2 * x[:-k] ** 2) / s2 ** 2 for k in ks])
        q = acorr_ljungbox(rets[a], lags=[m_q], return_df=True)
        qt = float(np.sum(rho[:m_q] ** 2 / tau[:m_q]))
        rows.append({'asset': LABELS[a], 'T': T, 'Q10': float(q['lb_stat'].iloc[0]), 'Q10_p': float(q['lb_pvalue'].iloc[0]),
                     'Qt10': qt, 'Qt10_p': float(stats.chi2.sf(qt, m_q)),
                     'sig_classic': int(np.sum(np.abs(rho) > 1.96 / np.sqrt(T))),
                     'sig_robust': int(np.sum(np.abs(rho) > 1.96 * np.sqrt(tau))),
                     'robust_lags': [int(k) for k in ks[np.abs(rho) > 1.96 * np.sqrt(tau)]], 'rho1': rho[0]})
    return pd.DataFrame(rows).set_index('asset')


def fig_robust_acf(asset='sp500', m=20):
    x = (rets[asset] - rets[asset].mean()).values
    T, s2 = len(x), np.sum(x ** 2)
    ks = np.arange(1, m + 1)
    rho = np.array([np.sum(x[k:] * x[:-k]) / s2 for k in ks])
    tau = np.array([np.sum(x[k:] ** 2 * x[:-k] ** 2) / s2 ** 2 for k in ks])
    fig, ax = plt.subplots(figsize=(6.8, 2.9))
    ax.bar(ks, rho, color=MainBlue, width=0.6, label='$\\hat\\rho_k$')
    ax.plot(ks, 1.96 * np.sqrt(tau), color=IDAred, lw=1.1, label='Robust band $\\pm1.96\\sqrt{\\hat\\tau_k}$')
    ax.plot(ks, -1.96 * np.sqrt(tau), color=IDAred, lw=1.1)
    ax.axhline(1.96 / np.sqrt(T), color=Gray, ls='--', lw=0.9, label='Classical band $\\pm1.96/\\sqrt{T}$')
    ax.axhline(-1.96 / np.sqrt(T), color=Gray, ls='--', lw=0.9)
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_xticks(ks[::2] if m > 10 else ks)
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel('Autocorrelation of $r_t$')
    ax.set_title(f'{LABELS[asset]}: classical vs heteroskedasticity-robust bands', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=3, y=-0.25)
    plt.tight_layout()
    save_fig('ch1_sem_robust_acf')


def fig_kurtosis_expanding(n_paths=500, nus=(3, 6)):
    """Excesul de kurtosis al S&P 500 pe primii n ani (n = 1..36) vs benzi 5-95% din Student-t simulate."""
    r = rets['sp500']
    first = r.index[0]
    ends = np.array([np.searchsorted(r.index, first + pd.DateOffset(years=y)) for y in range(1, 37)])
    ends = ends[(ends > 0) & (ends <= len(r))]
    k_data = _kurt_path(r.values[:, None], ends)[:, 0]
    rng = np.random.default_rng(SEED)
    bands = {}
    for nu in nus:
        sims = stats.t.rvs(nu, size=(len(r), n_paths), random_state=rng)
        K = _kurt_path(sims, ends)
        bands[nu] = (np.quantile(K, 0.05, axis=1), np.quantile(K, 0.5, axis=1), np.quantile(K, 0.95, axis=1))
    years = np.arange(1, len(ends) + 1)
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    for nu, c in zip(nus, (IDAred, Forest)):
        lo, med, hi = bands[nu]
        ax.fill_between(years, lo, hi, color=c, alpha=0.15, lw=0, label=f'Student-t, $\\nu$={nu}: 5%-95% of {n_paths} paths')
        ax.plot(years, med, color=c, lw=0.8, ls='--', label=f'Student-t, $\\nu$={nu}: median')
    ax.plot(years, k_data, color=MainBlue, lw=1.4, marker='o', ms=2.5, label='S&P 500 (from January 1990)')
    ax.set_yscale('symlog', linthresh=10)
    ax.set_xlabel('Window length: first $n$ years of data')
    ax.set_ylabel('Excess kurtosis')
    ax.set_title('Expanding-window kurtosis: data vs simulated Student-t', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.3)
    plt.tight_layout()
    save_fig('ch1_sem_kurtosis_expanding')
    kd = pd.Series(k_data, index=years)
    jumps = kd.diff().sort_values(ascending=False).head(3)
    top_days = r.abs().sort_values(ascending=False).head(5)
    return dict(k_by_year=kd.round(2).to_dict(), jumps=jumps.round(2).to_dict(),
                top_days=[(d.date(), round(100 * r[d], 1)) for d in top_days.index],
                band3_end=(bands[3][0][-1], bands[3][1][-1], bands[3][2][-1]),
                band6_end=(bands[6][0][-1], bands[6][1][-1], bands[6][2][-1]),
                band6_width_y5=bands[6][2][4] - bands[6][0][4], band6_width_end=bands[6][2][-1] - bands[6][0][-1],
                band3_width_y5=bands[3][2][4] - bands[3][0][4], band3_width_end=bands[3][2][-1] - bands[3][0][-1],
                inside3=bool(bands[3][0][-1] <= kd.iloc[-1] <= bands[3][2][-1]),
                inside6=bool(bands[6][0][-1] <= kd.iloc[-1] <= bands[6][2][-1]))


def hill_tails(assets=('bettr', 'btc'), frac=0.02):
    rows = []
    for a in assets:
        r = rets[a]
        k = int(frac * len(r))
        aL, aR = hill_estimator(-r.values, k), hill_estimator(r.values, k)
        z = (r - r.mean()) / r.std()
        nu = stats.t.fit(z)[0]
        zdiff = (aL - aR) / np.sqrt(aL ** 2 / k + aR ** 2 / k)
        rows.append({'asset': LABELS[a], 'n': len(r), 'k': k, 'alpha_left': aL,
                     'ci_left': (aL * (1 - 1.96 / np.sqrt(k)), aL * (1 + 1.96 / np.sqrt(k))),
                     'alpha_right': aR,
                     'ci_right': (aR * (1 - 1.96 / np.sqrt(k)), aR * (1 + 1.96 / np.sqrt(k))),
                     'z_diff': zdiff, 'p_diff': 2 * stats.norm.sf(abs(zdiff)), 'nu_mle': nu})
    return pd.DataFrame(rows).set_index('asset')


def fig_hill_tails(assets=('bettr', 'btc')):
    fig, axes = plt.subplots(1, len(assets), figsize=(7.2, 3.0), sharey=True)
    for ax, a in zip(axes, assets):
        r = rets[a].values
        for side, x, c, lab in [('left', -r, IDAred, 'Left tail (losses)'), ('right', r, Forest, 'Right tail (gains)')]:
            ks = np.array([max(int(f * len(r)), 5) for f in HILL_FRACS_TAILS])
            al = np.array([hill_estimator(x, k) for k in ks])
            ax.plot(100 * HILL_FRACS_TAILS, al, color=c, lw=1.1, label=lab)
            ax.fill_between(100 * HILL_FRACS_TAILS, al * (1 - 1.96 / np.sqrt(ks)), al * (1 + 1.96 / np.sqrt(ks)),
                            color=c, alpha=0.15, lw=0)
        z = (rets[a] - rets[a].mean()) / rets[a].std()
        ax.axhline(stats.t.fit(z)[0], color=Gray, ls='--', lw=0.9, label='Student-t $\\hat\\nu$ (MLE)')
        ax.axhline(4, color=LightGray, lw=0.8)
        ax.set_title(LABELS[a], fontsize=9, loc='left')
        ax.set_xlabel('Tail fraction $k/n$ (%)')
    axes[0].set_ylabel('Hill tail index $\\hat\\alpha$')
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=3, frameon=False)
    plt.tight_layout()
    save_fig('ch1_sem_hill_tails')


def fig_leverage_ro_crypto(K=20):
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    out = {}
    for a in ('bettr', 'btc'):
        ks, vals = leverage_corr(rets[a], K)
        L = pd.Series(vals, index=ks)
        out[a] = L
        ax.plot(L.index, L.values, marker='o', ms=2.5, lw=1.0, color=COLORS[a], label=LABELS[a])
    ax.axhline(0, color=Gray, lw=0.5)
    ax.axvline(0, color=LightGray, lw=0.5)
    band = 1.96 / np.sqrt(len(rets['bettr']))
    ax.axhspan(-band, band, color=LightGray, alpha=0.5, lw=0)
    ax.set_xlabel('Lag $k$ (days); $k>0$: return today, volatility later')
    ax.set_ylabel('corr$(r_t, |r_{t+k}|)$')
    ax.set_title('Leverage effect: Romania (BET-TR) vs crypto (Bitcoin)', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    plt.tight_layout()
    save_fig('ch1_sem_leverage_ro_crypto')
    return pd.DataFrame(out)


def leverage_regression(asset, n_controls=5):
    """|r_{t+1}| = c + b_plus r_t^+ + b_minus r_t^- + sum_{j=1..5} d_j |r_{t-j}| + u, erori Newey-West (HAC).

    |r_t| nu intra printre controale: |r_t| = r_t^+ - r_t^- (coliniaritate perfecta)."""
    r = rets[asset]
    df = pd.DataFrame({'y': r.abs().shift(-1), 'pos': np.maximum(r, 0), 'neg': np.minimum(r, 0)})
    for j in range(1, n_controls + 1):
        df[f'abs_lag{j}'] = r.abs().shift(j)
    df = df.dropna()
    L = int(4 * (len(df) / 100) ** (2 / 9))
    fit = sm.OLS(df['y'], sm.add_constant(df.drop(columns='y'))).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    w = fit.wald_test('pos + neg = 0', scalar=True)
    return dict(b_pos=fit.params['pos'], se_pos=fit.bse['pos'], b_neg=fit.params['neg'], se_neg=fit.bse['neg'],
                wald=float(w.statistic), wald_p=float(w.pvalue), lags=L, n=len(df))


def bettr_vs_sp500(n_boot=2000, block=21):
    px = pd.concat([closes['bettr'], closes['sp500']], axis=1, keys=['bet', 'spx']).dropna()   # join pe PRETURI
    r = np.log(px).diff().dropna()
    T = len(r)
    sr = r.mean() / r.std()                                   # Sharpe zilnic, r_f = 0
    rho = r['bet'].corr(r['spx'])
    theta = 2 - 2 * rho + 0.5 * (sr['bet'] ** 2 + sr['spx'] ** 2 - 2 * sr['bet'] * sr['spx'] * rho ** 2)
    z_jkm = (sr['bet'] - sr['spx']) / np.sqrt(theta / T)
    # bootstrap pe blocuri (blocuri circulare de 21 de zile) pentru diferenta Sharpe anualizata
    rng = np.random.default_rng(SEED)
    vals = r.values
    nb = int(np.ceil(T / block))
    diffs = []
    for _ in range(n_boot):
        starts = rng.integers(0, T, nb)
        idx = (starts[:, None] + np.arange(block)[None, :]).ravel()[:T] % T
        s = vals[idx]
        m, sd = s.mean(0), s.std(0, ddof=1)
        diffs.append(np.sqrt(252) * (m[0] / sd[0] - m[1] / sd[1]))
    diffs = np.array(diffs)
    d_obs = np.sqrt(252) * (sr['bet'] - sr['spx'])
    # corelatii cu decalaj: BET ziua t vs S&P ziua t-1 (informatia americana ajunge a doua zi la BVB)
    lag1 = r['bet'].corr(r['spx'].shift(1))
    lead1 = r['bet'].corr(r['spx'].shift(-1))
    wk = np.log(px.resample('W-FRI').last()).diff().dropna()
    return dict(start=px.index[0].date(), end=px.index[-1].date(), T=T,
                sr_ann_bet=np.sqrt(252) * sr['bet'], sr_ann_spx=np.sqrt(252) * sr['spx'],
                diff_ann=d_obs, rho=rho, z_jkm=z_jkm, p_jkm=2 * stats.norm.sf(abs(z_jkm)),
                boot_ci=(np.quantile(diffs, 0.025), np.quantile(diffs, 0.975)),
                boot_p=2 * min((diffs <= 0).mean(), (diffs >= 0).mean()),
                corr_lag0=rho, corr_lag1=lag1, corr_lead1=lead1, corr_sw=rho + lag1,
                corr_weekly=wk['bet'].corr(wk['spx']))


def garch_check(n_paths=200, lags=(1, 20, 100), frac=0.02):
    """GARCH(1,1)-t pe S&P 500; 200 traiectorii simulate; benzi 5-95% pentru fapte stilizate."""
    from arch import arch_model
    r = 100 * rets['sp500']
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
    res = am.fit(disp='off')
    n = len(r)
    k = int(frac * n)

    def facts(x):
        x = pd.Series(np.asarray(x))
        a = x.abs()
        return dict(exkurt=stats.kurtosis(x), **{f'acf{j}': a.autocorr(j) for j in lags},
                    hill=hill_estimator(a.values, k), L1=x.corr(a.shift(-1)))

    data = facts(r.values)
    sims = [facts(am.simulate(res.params, nobs=n, burn=1000)['data'].values) for _ in range(n_paths)]
    sims = pd.DataFrame(sims)
    q05, q50, q95 = sims.quantile(0.05), sims.quantile(0.5), sims.quantile(0.95)
    table = pd.DataFrame({'data': pd.Series(data), 'q05': q05, 'median': q50, 'q95': q95})
    table['reproduced'] = (table['data'] >= table['q05']) & (table['data'] <= table['q95'])
    # grafic: ACF |r| date vs 20 de traiectorii
    K = np.arange(1, 251)
    data_curve = [r.abs().autocorr(j) for j in K]
    curves = []
    for _ in range(20):
        s_ = pd.Series(am.simulate(res.params, nobs=n, burn=1000)['data'].values).abs()
        curves.append([s_.autocorr(j) for j in K])
    curves = np.array(curves)
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    ax.fill_between(K, np.quantile(curves, 0.05, axis=0), np.quantile(curves, 0.95, axis=0), color=IDAred,
                    alpha=0.15, lw=0, label='GARCH(1,1)-t simulations: 5%-95%')
    ax.plot(K, np.median(curves, axis=0), color=IDAred, lw=1.1, label='GARCH(1,1)-t simulations: median')
    ax.plot(K, data_curve, color=MainBlue, lw=1.1, label='S&P 500 data')
    ax.set_xscale('log')
    ax.set_xlabel('Lag $k$ (days, log scale)')
    ax.set_ylabel('ACF of $|r_t|$')
    ax.set_title('Does GARCH(1,1)-t reproduce the long memory of volatility?', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    plt.tight_layout()
    save_fig('ch1_sem_garch_check')
    return dict(params=res.params.round(4).to_dict(), persistence=res.params['alpha[1]'] + res.params['beta[1]'],
                table=table, n_paths=n_paths, k=k)


def traded_value_proxy():
    """Valoarea zilnica tranzactionata (lei) a acțiunilor blue-chip BVB din data/market: volum x pret de inchidere."""
    v = []
    for sym in BLUE_CHIPS:
        d = read_market(sym)
        v.append((d['volume'] * d['close']).rename(sym))
    return pd.concat(v, axis=1).sum(axis=1, min_count=1)


def fig_bettr_rolling_acf(window=250, split='2020-09-21'):
    r = rets['bettr']
    vals, lo, hi, idx = [], [], [], []
    for end in range(window, len(r) + 1, 5):
        w = r.iloc[end - window:end]
        rho, se = rho_robust(w)
        vals.append(rho); lo.append(rho - 1.96 * se); hi.append(rho + 1.96 * se); idx.append(w.index[-1])
    roll = pd.Series(vals, index=idx)
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    ax.plot(idx, vals, color=IDAred, lw=1.0, label='Rolling lag-1 autocorrelation (250 days)')
    ax.fill_between(idx, lo, hi, color=IDAred, alpha=0.15, lw=0, label='95% band, heteroskedasticity-robust')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.axvline(pd.Timestamp(split), color=MainBlue, ls='--', lw=0.9, label='FTSE Russell reclassification (Sep 2020)')
    ax.set_ylabel('$\\hat\\rho_1$')
    ax.set_title('BET-TR: did the lag-1 autocorrelation fall after the upgrade?', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.13)
    plt.tight_layout()
    save_fig('ch1_sem_bettr_rolling_acf')
    pre, post = r.loc[:split].iloc[:-1], r.loc[split:]
    rp, sp = rho_robust(pre)
    rq, sq = rho_robust(post)
    z = (rp - rq) / np.sqrt(sp ** 2 + sq ** 2)
    df = pd.DataFrame({'y': r, 'x': r.shift(1)}).dropna()
    df['post'] = (df.index >= pd.Timestamp(split)).astype(float)
    df['x_post'] = df['x'] * df['post']
    fit = sm.OLS(df['y'], sm.add_constant(df[['x', 'post', 'x_post']])).fit(cov_type='HAC', cov_kwds={'maxlags': 10})
    # lichiditate: valoarea tranzactionata a blue-chip-urilor, medie mobila pe 250 de zile (log)
    tv = traded_value_proxy().reindex(r.index)
    liq = np.log(tv.rolling(window, min_periods=200).mean()).reindex(roll.index)
    ok = liq.notna()
    corr_liq = float(np.corrcoef(roll[ok], liq[ok])[0, 1])
    tv_pre, tv_post = tv.loc[pre.index].median(), tv.loc[post.index].median()
    return dict(pre=(rp, sp, len(pre), pre.index[0].date(), pre.index[-1].date()),
                post=(rq, sq, len(post), post.index[0].date(), post.index[-1].date()),
                z=z, p=2 * stats.norm.sf(abs(z)), hac_diff=fit.params['x_post'], hac_se=fit.bse['x_post'],
                hac_p=fit.pvalues['x_post'], share_sig=float(np.mean(np.array(lo) > 0)),
                corr_rho_liquidity=corr_liq, tv_median_pre_mn=tv_pre / 1e6, tv_median_post_mn=tv_post / 1e6)

## Run

In [ ]:
print(robust_lb_table().round(3))
fig_robust_acf()
print(fig_kurtosis_expanding())
print(hill_tails().T)
fig_hill_tails()
print(fig_leverage_ro_crypto().loc[[1, 5]].round(3))
for a in ('sp500', 'bettr', 'btc', 'eurron'):
    print(a, leverage_regression(a))
print(bettr_vs_sp500())
gc = garch_check()
print(gc['table'].round(3))
print(fig_bettr_rolling_acf())

![ch1_sem_robust_acf](./ch1_sem_robust_acf.png)

Output: `ch1_sem_robust_acf.pdf`

![ch1_sem_kurtosis_expanding](./ch1_sem_kurtosis_expanding.png)

Output: `ch1_sem_kurtosis_expanding.pdf`

![ch1_sem_hill_tails](./ch1_sem_hill_tails.png)

Output: `ch1_sem_hill_tails.pdf`

![ch1_sem_leverage_ro_crypto](./ch1_sem_leverage_ro_crypto.png)

Output: `ch1_sem_leverage_ro_crypto.pdf`

![ch1_sem_garch_check](./ch1_sem_garch_check.png)

Output: `ch1_sem_garch_check.pdf`

![ch1_sem_bettr_rolling_acf](./ch1_sem_bettr_rolling_acf.png)

Output: `ch1_sem_bettr_rolling_acf.pdf`